In [0]:
# /Volumes/retail_lakehouse/bronze/landingzone

In [0]:
from pyspark.sql import functions as F

In [0]:
volume_root = "/Volumes/retail_lakehouse/bronze/landingzone"
checkpoint_root = '/Volumes/retail_lakehouse/bronze/landingzone/_checkpoints'

In [0]:
# json data ingestion - auto loader

order_json_path = f'{volume_root}'

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","json")
    .option("cloudFiles.schemaLocation",f"{checkpoint_root}/orders_stream")
    .option("cloudFiles.inferColumnTypes","true")
    .load(order_json_path)
    .withColumn("_ingested_at",F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
# trigger - executes increamental batch
(
    orders_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation",f"{checkpoint_root}/orders_delta")
    .trigger(availableNow=True)
    .toTable("retail_lakehouse.bronze.orders_raw")
 )
print("order json streaming ingestion completed.....")

In [0]:
customers_parquet_path = f"{volume_root}/customers.parquet"
customer_df = (
    spark.read.format("parquet")
    .load(customers_parquet_path)
    .withColumn("_ingested_at",F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)
customer_df.write.format("delta").mode("append").saveAsTable("retail_lakehouse.bronze.customers_raw")

In [0]:
products_csv_path = f"{volume_root}/products.csv"

products_df = (
    spark.read.format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(products_csv_path)
    .withColumn("_ingested_at",F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

products_df.write.format("delta").mode("append").saveAsTable("retail_lakehouse.bronze.products_raw")

In [0]:
%sql
DESCRIBE HISTORY retail_lakehouse.bronze.orders_raw;